# DoChat — RAG Pipeline Deep Dive

This notebook explains and demonstrates the Retrieval-Augmented Generation pipeline behind DoChat, from document chunking through FAISS indexing to LLM-powered Q&A.

In [ ]:
# !pip install langchain langchain-community sentence-transformers faiss-cpu

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded ✓")

## 1. Document Chunking

We split documents into overlapping chunks so the retriever can find precise passages rather than returning entire documents.

In [ ]:
sample_doc = """
Artificial intelligence (AI) is transforming data analytics at an unprecedented pace.
Machine learning models can now process millions of records in seconds.
Natural language processing allows analysts to query databases using plain English.
Large language models like GPT-4 and Claude can explain complex statistical findings
in language that non-technical stakeholders can understand.
Power BI and Tableau are being enhanced with AI features for automated insight generation.
Python remains the dominant language for data science, with pandas and scikit-learn
forming the core of most analytics workflows.
SQL continues to be essential for data extraction and transformation in enterprise systems.
"""

def chunk_text(text, chunk_size=150, overlap=30):
    """Sliding window chunking with character overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
    return chunks

chunks = chunk_text(sample_doc)
print(f"Document → {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  Chunk {i+1} ({len(c)} chars): {c[:80]}...")

## 2. Embedding & Semantic Search

We convert chunks into dense vector embeddings and use cosine similarity for retrieval.

In [ ]:
# Lightweight TF-IDF demo (sentence-transformers not required)
docs = [
    "Python pandas dataframes are used for data manipulation and analysis.",
    "SQL window functions like ROW_NUMBER and LAG enable advanced analytics.",
    "Power BI dashboards provide real-time operational visibility for leadership.",
    "FAISS is a library for efficient similarity search in high-dimensional spaces.",
    "LangChain agents can use tools to query databases and APIs autonomously.",
    "XGBoost achieves state-of-the-art results on tabular classification tasks.",
    "Azure Data Factory orchestrates ETL pipelines between cloud data sources.",
]

query = "How do I query data with Python?"

vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words='english')
all_texts  = [query] + docs
tfidf      = vectorizer.fit_transform(all_texts)

sims = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
ranked = sorted(zip(sims, docs), reverse=True)

print(f"Query: '{query}'")
print("\nTop-3 Retrieved Chunks:")
for i, (score, doc) in enumerate(ranked[:3]):
    print(f"  [{i+1}] score={score:.3f}  |  {doc}")

## 3. RAG Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)
ax.axis('off')
fig.patch.set_facecolor('#0c0f1a')
ax.set_facecolor('#0c0f1a')

boxes = [
    (0.3, 3.5, 1.8, 'Documents\n(Any text)', '#7c3aed'),
    (2.5, 3.5, 1.8, 'Chunking\n(500 chars,\n50 overlap)', '#7c3aed'),
    (4.7, 3.5, 1.8, 'Embeddings\n(MiniLM-L6)', '#7c3aed'),
    (6.9, 3.5, 1.8, 'FAISS Index\n(Vector DB)', '#7c3aed'),
    (0.3, 1.0, 1.8, 'User Query', '#059669'),
    (2.5, 1.0, 1.8, 'Query\nEmbedding', '#059669'),
    (4.7, 1.0, 1.8, 'Top-K\nRetrieval', '#059669'),
    (6.9, 1.0, 1.8, 'LLM\n(Mistral-7B)', '#059669'),
    (9.2, 1.0, 1.8, 'Answer +\nCitations', '#e11d48'),
]

for (x, y, w, label, color) in boxes:
    rect = mpatches.FancyBboxPatch((x, y), w, 1.0, boxstyle='round,pad=0.1',
                                    facecolor=color+'33', edgecolor=color, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x + w/2, y + 0.5, label, ha='center', va='center',
            color='white', fontsize=8, fontweight='bold')

arrows_index = [(1), (2), (3)]  # top row arrows
arrows_query = [(4), (5), (6), (7), (8)]  # bottom row
for i, (x, y, w, _, color) in enumerate(boxes[:-1]):
    if i < 3:
        ax.annotate('', xy=(boxes[i+1][0], boxes[i+1][1]+0.5),
                    xytext=(x+w, y+0.5),
                    arrowprops=dict(arrowstyle='->', color='#64748b', lw=1.5))
    elif i >= 4 and i < 8:
        ax.annotate('', xy=(boxes[i+1][0], boxes[i+1][1]+0.5),
                    xytext=(x+w, y+0.5),
                    arrowprops=dict(arrowstyle='->', color='#64748b', lw=1.5))

# Vertical arrow from FAISS to Retrieval
ax.annotate('', xy=(5.8, 2.0), xytext=(7.8, 3.5),
            arrowprops=dict(arrowstyle='->', color='#a78bfa', lw=1.5, linestyle='dashed'))

ax.text(6.0, 4.7, 'INDEX PHASE', color='#a78bfa', fontsize=9, fontweight='bold')
ax.text(6.0, 0.6, 'QUERY PHASE', color='#34d399', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../diagrams/rag_architecture.png', dpi=120, bbox_inches='tight', facecolor='#0c0f1a')
plt.show()
print("Saved to diagrams/rag_architecture.png")